# The Relaxation Fingerprint: Reading Protein Motion from NMR

A protein is never truly still. Even a "rigid" α-helix experiences constant thermal fluctuations
on the picosecond–nanosecond timescale. NMR **relaxation** measurements are the premier tool for
characterizing this motion directly in solution, at atomic resolution, without disturbing the protein.

Three rates tell the complete story:

| Rate | Symbol | Physical meaning |
|---|---|---|
| Longitudinal | R₁ (s⁻¹) | How fast magnetization recovers along B₀ |
| Transverse | R₂ (s⁻¹) | How fast coherence is lost in the xy-plane |
| Heteronuclear NOE | hetNOE | Sensitivity to fast (ps–ns) local motion |

> **The key**: rigid residues have **high R₂** and **high hetNOE**. Flexible loops have
> **low R₂** and **low hetNOE**. Reading the pattern reveals which parts of the protein
> move and which are anchored.

In [ ]:
import os
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !pip install -q synth-pdb synth-nmr biotite matplotlib numpy
else:
    sys.path.insert(0, os.path.abspath("../../"))

import io
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import biotite.structure.io.pdb as bpdb
from synth_pdb.generator import generate_pdb_content
from synth_nmr import calculate_relaxation_rates

print("✅ Environment ready")

## The Lipari-Szabo Model-Free Formalism

The standard framework for interpreting relaxation data is the **Lipari-Szabo model-free approach**
(1982 — one of the most-cited papers in NMR). It describes backbone N-H motion with two parameters:

- **S²** (order parameter): 0 = completely disordered, 1 = completely rigid
- **τ_c** (correlation time): the timescale of global molecular tumbling (ns range)

The **R₂/R₁ ratio** is remarkably useful — it's largely independent of the internal dynamics
and primarily reflects τ_c. This lets you estimate the global tumbling time of the molecule
from a single measurement, and identify residues that exchange between conformations
(they show anomalously high R₂ due to "chemical exchange").

In [ ]:
UBIQUITIN_SEQ = "MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSDYNIQKESTLHLVLRLRGG"

print("Generating ubiquitin structure...")
pdb_str = generate_pdb_content(sequence_str=UBIQUITIN_SEQ, conformation="alpha", seed=42)
struct = bpdb.PDBFile.read(io.StringIO(pdb_str)).get_structure(model=1)
print(f"  Structure: {len(UBIQUITIN_SEQ)} residues, {struct.array_length()} atoms")

print("\nCalculating backbone relaxation rates at two field strengths...")
# tau_m = 4 ns for a 76-aa protein at 298 K (Stokes-Einstein estimate)
r600 = calculate_relaxation_rates(struct, field_mhz=600.0, tau_m_ns=4.0)
r900 = calculate_relaxation_rates(struct, field_mhz=900.0, tau_m_ns=4.0)

res_ids = sorted(r600.keys())


def extract(data, key):
    return np.array([data[r][key] for r in res_ids])


R1_600 = extract(r600, "R1")
R2_600 = extract(r600, "R2")
NOE_600 = extract(r600, "NOE")
S2_600 = extract(r600, "S2")
R1_900 = extract(r900, "R1")
R2_900 = extract(r900, "R2")

print("\n📊 Summary at 600 MHz:")
print(f"   R1 : {R1_600.mean():.2f} ± {R1_600.std():.2f} s⁻¹")
print(f"   R2 : {R2_600.mean():.2f} ± {R2_600.std():.2f} s⁻¹")
print(f"   NOE: {NOE_600.mean():.3f} ± {NOE_600.std():.3f}")
print(f"   S2 : {S2_600.mean():.3f} ± {S2_600.std():.3f}")

## The Three-Panel Relaxation Fingerprint

Plotting R₁, R₂, and hetNOE as a function of residue number creates the **relaxation fingerprint** —
the single most informative NMR dynamics plot:

- **Spikes in R₂** → rigid secondary structure elements or chemical exchange
- **Dips in hetNOE** → flexible loops and termini
- **Dips in R₁** → either very fast or very slow motions relative to the spectrometer frequency

In [ ]:
plt.style.use("seaborn-v0_8-darkgrid")
fig, axes = plt.subplots(3, 1, figsize=(13, 10), sharex=True)

data_panels = [
    (R1_600, "R₁ (s⁻¹)", "#4cc9f0", "Longitudinal Relaxation Rate"),
    (R2_600, "R₂ (s⁻¹)", "#f72585", "Transverse Relaxation Rate"),
    (NOE_600, "hetNOE", "#7bed9f", "Heteronuclear NOE"),
]

for ax, (vals, ylabel, color, title) in zip(axes, data_panels):
    ax.fill_between(res_ids, vals, alpha=0.22, color=color)
    ax.plot(res_ids, vals, color=color, lw=2.0, zorder=3)
    ax.axhline(
        vals.mean(), color="white", lw=1.2, ls="--", alpha=0.7, label=f"Mean = {vals.mean():.2f}"
    )
    ax.set_ylabel(ylabel, fontsize=11)
    ax.set_title(title, fontsize=12, fontweight="bold", pad=4)
    ax.legend(fontsize=10, loc="upper right")
    ax.set_xlim(res_ids[0], res_ids[-1])

# Add hetNOE reference line at 0.65 (typical helix threshold)
axes[2].axhline(0.65, color="orange", lw=1.0, ls=":", alpha=0.8, label="Rigid threshold (0.65)")
axes[2].legend(fontsize=10, loc="upper right")

axes[-1].set_xlabel("Residue Number", fontsize=12)
fig.suptitle(
    "NMR Relaxation Fingerprint of Ubiquitin @ 600 MHz", fontsize=15, fontweight="bold", y=1.01
)
plt.tight_layout()
plt.savefig("relaxation_fingerprint.png", dpi=150, bbox_inches="tight")
plt.show()

## Field Dependence: 600 MHz vs 900 MHz

Different NMR observables respond differently to field strength, which is a powerful diagnostic:

- **R₁** peaks at a field where ω₀τ_c ≈ 1 (the "dispersion" condition)
- **R₂** generally **increases** at higher fields (chemical shift anisotropy contribution grows as B₀²)
- The difference between R₂ at two fields directly reports on slow (μs–ms) conformational exchange

This is why NMR facilities with multiple magnets (600, 800, 900 MHz) are so valuable — the
**field-dependence pattern** tells you *what kind* of motion is present.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

x = np.arange(len(res_ids))
w = 0.38

# R2 grouped bar chart
ax1.bar(x - w / 2, R2_600, width=w, color="#f72585", alpha=0.85, label="600 MHz", edgecolor="none")
ax1.bar(x + w / 2, R2_900, width=w, color="#b5179e", alpha=0.85, label="900 MHz", edgecolor="none")
ax1.set_xticks(x[::5])
ax1.set_xticklabels([str(res_ids[i]) for i in range(0, len(res_ids), 5)], fontsize=9)
ax1.set_xlabel("Residue Number", fontsize=11)
ax1.set_ylabel("R₂ (s⁻¹)", fontsize=11)
ax1.set_title(
    "R₂ at 600 vs 900 MHz\nField-Dependence Reveals Exchange Dynamics",
    fontsize=12,
    fontweight="bold",
)
ax1.legend(fontsize=11)

# R2/R1 ratio scatter coloured by NOE
ratio = R2_600 / np.where(R1_600 > 0, R1_600, 1.0)
sc = ax2.scatter(
    res_ids, ratio, c=NOE_600, cmap="RdYlGn", s=55, zorder=3, vmin=NOE_600.min(), vmax=NOE_600.max()
)
ax2.axhline(
    ratio.mean(),
    color="navy",
    lw=1.5,
    ls="--",
    label=f"Mean R₂/R₁ = {ratio.mean():.1f}  →  τ_c ≈ 4 ns",
)
plt.colorbar(sc, ax=ax2, label="hetNOE value")
ax2.set_xlabel("Residue Number", fontsize=11)
ax2.set_ylabel("R₂ / R₁", fontsize=11)
ax2.set_title(
    "R₂/R₁ Ratio (Global Tumbling Probe)\nColoured by hetNOE (green=rigid, red=flexible)",
    fontsize=12,
    fontweight="bold",
)
ax2.legend(fontsize=10)
ax2.set_xlim(res_ids[0] - 1, res_ids[-1] + 1)

plt.tight_layout()
plt.savefig("relaxation_field_dependence.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Mean R₂/R₁ = {ratio.mean():.2f}  →  estimated τ_c ≈ 4 ns (consistent with input)")

## From Relaxation to Biology

The relaxation fingerprint is not just a technical measurement — it connects directly to function:

- **Ubiquitin's flexible C-terminal tail** (residues 73–76): the low hetNOE here reflects genuine
  flexibility that E2 ubiquitin-conjugating enzymes exploit to bind and transfer ubiquitin chains.

- **The β-grasp core** (residues 1–72): high R₂ and hetNOE throughout this region reflects the
  rigid scaffold that makes ubiquitin an ultra-stable tag for proteasomal targeting.

The rigid core survives extreme conditions; the flexible tail provides the molecular handle.
This division of labor — revealed by relaxation measurements — is why ubiquitin can function
as a universal signal across all eukaryotic life.